In [3]:

# Import
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd


In [4]:
# Task 01
def log_file_reader(file_path: str | Path) -> tuple[str, str]:
    """Leser en fil ved hjælp av pathlib.Path og automatisk håndtering av UTF-8 eller ISO-8859-1.

    Returnerer: (file_text_content, encoding_used)
    """
    path = Path(file_path)

    try:
        content = path.read_text(encoding="utf-8")
        return content, "utf-8"
    except UnicodeDecodeError:
        content = path.read_text(encoding="ISO-8859-1")
        return content, "ISO-8859-1"
    
    
def parse_log_lines(file_content: str) -> list[dict]:
    """Parser gyldige salgslinjer ved hjelp av regex.

    Returnerer en liste med dictionaries for hver gyldige transaksjon.
    """
    pattern = re.compile(
        r"^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s+"
        r"\[(?P<register_id>[A-Za-z0-9_]+)\]\s+"
        r"item=(?P<item>[^\s=]+)\s+"
        r"price=(?P<price>\d+(?:\.\d+)?)\s+"
        r"(?P<currency>EUR|NOK|USD)\s+"
        r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)$"
    )

    records = []

    for line in file_content.splitlines():
        line = line.strip()
        match = pattern.match(line)

        if match:
            record = {
                "timestamp": match.group("timestamp"),
                "register_id": match.group("register_id"),
                "item": match.group("item"),
                "price": float(match.group("price")),
                "currency": match.group("currency"),
                "status": match.group("status"),
            }
            records.append(record)
    return records

# Søker etter første 'register_sales.log'-fil i prosjektmappen
log_files = list(Path.cwd().rglob("register_sales.log"))

if log_files:
    file_path1 = log_files[0]  # Velger f.eks. POS_01/register_sales.log
    content, encoding = log_file_reader(file_path1)
    records = parse_log_lines(content)

    print(f"Fant og leste filen: {file_path1}")
    print(f"Tegnsett brukt: {encoding}")
    print(f"Antall transaksjoner funnet: {len(records)}")
else:
    print(
        "Fant ingen loggfil. Pass på at du kjører koden fra prosjektmappen."
    )



Fant og leste filen: c:\Users\Bruker\Documents\INF201\Koding INF201\INF201_group_project\d2_data\2026\student_union\POS_01\register_sales.log
Tegnsett brukt: utf-8
Antall transaksjoner funnet: 80


In [5]:
# Task 02

def directory_search_log_files(directory: str | Path) -> list[Path]:
    """Søker rekursivt etter alle .log-filer i et mappehierarki.

    Returnerer en liste med pathlib.Path-objekter.
    """
    path = Path(directory)
    log_files = []

    if not path.exists() or not path.is_dir():
        return log_files
    
    for item in path.iterdir():
        if item.is_dir():
            log_files.extend(directory_search_log_files(item)) # Rekursivt kall
        elif item.is_file() and item.suffix == ".log":
            log_files.append(item)

    return log_files 


def metadata_recursive(metadata: dict | list) -> set[str]:
    """ Traverserer nøstede ordbøker og lister rekursivt for å hente ut alle
    register-IDer hvor "active": true. 
    Returnerer et set[str]. 
    """
    active_stores= set()

    if isinstance(metadata, dict):
        if(
            "id" in metadata 
            and "active" in metadata
            and metadata["active"] is True 
            and isinstance(metadata["id"], str)
        ):
            active_stores.add(metadata["id"])

        for value in metadata.values():
            active_stores.update(metadata_recursive(value))

    elif isinstance(metadata, list):
        for item in metadata:
            active_stores.update(metadata_recursive(item))

    return active_stores 






In [6]:
# Task 03

# ==============================================================================
# Task 3.1: Function for removing price outliers
# ==============================================================================
def clean_price_outliers(prices: list | np.ndarray) -> np.ndarray:
    """
    Konverterer priser til et 1D NumPy float-array, identifiserer uvanlige 
    hardware-glitch priser (999.00) som avviker mer enn 3 standardavvik fra 
    gjennomsnittet (|x - mu| > 3*sigma), og erstatter dem med medianen.
    """
    # 1. Konverter til 1D NumPy float-array
    price_array = np.array(prices, dtype=float)
    
    # 2. Beregn gjennomsnitt og standardavvik
    mean = np.mean(price_array)
    std = np.std(price_array)
    
    # 3. Detekter maskinvarefeil (999.00) som avviker med > 3 std fra gjennomsnittet
    outlier_mask = (price_array == 999.00) & (np.abs(price_array - mean) > 3 * std)
    
    # 4. Beregn median for de gyldige (ikke-outlier) prisene
    valid_prices = price_array[~outlier_mask]
    median_val = np.median(valid_prices) if len(valid_prices) > 0 else 0.0
    
    # 5. Erstatt outliers med medianen og returner 1D-arrayet
    cleaned_array = price_array.copy()
    cleaned_array[outlier_mask] = median_val
    
    return cleaned_array


# ==============================================================================
# Task 3.2: Function for computing refund ratio
# ==============================================================================
def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """
    Beregner element-vis refund ratio ved formelen: Refund Count / (Sales Count + 10^-6).
    Tillegget 10^-6 (epsilon) forhindrer division-by-zero feil.
    """
    sales = np.array(sales_count, dtype=float)
    refunds = np.array(refund_count, dtype=float)
    
    epsilon = 1e-6
    ratio = refunds / (sales + epsilon)
    
    return ratio.round(3)



In [7]:
#Task 04

# Importerer/definerer compute_refund_ratio fra Oppgave 3.2 slik at Task 4.4 kan kalle den[cite: 5, 8]
def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    sales = np.array(sales_count, dtype=float)
    refunds = np.array(refund_count, dtype=float)
    return refunds / (sales + 1e-6)


# ==============================================================================
# Task 4.1: Convert list of dicts to DataFrame with DatetimeIndex
# ==============================================================================
def create_transaction_dataframe(transactions: list[dict]) -> pd.DataFrame:
    """
    Konverterer en liste med transaksjonsordbøker til pandas.DataFrame[cite: 5, 8].
    Konverterer timestamp til datetime, setter det som indeks og sorterer kronologisk[cite: 5, 8].
    """
    df = pd.DataFrame(transactions)
    
    # Konverter timestamp-kolonnen til datetime og sett som indeks[cite: 5, 8]
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df.set_index('timestamp', inplace=True)
    
    # Sorter DataFrame kronologisk[cite: 5, 8]
    df.sort_index(inplace=True)
    
    return df


# ==============================================================================
# Task 4.2: Group by item and resample in 1-hour windows (1h)
# ==============================================================================
def resample_hourly_revenue(df: pd.DataFrame) -> pd.DataFrame:
    """
    Grupperer data per vare (item) og resampler i 1-times vinduer (1h), 
    og beregner summen av pris per time.
    """
    hourly = (
        df.groupby(['item', pd.Grouper(freq='1h')])['price']
        .sum()
        .reset_index()
    )
    return hourly


# ==============================================================================
# Task 4.3: Pivot table of total revenue grouped by item across status
# ==============================================================================
def create_status_pivot_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Lager en pivot-tabell over total transaksjonsomsetning gruppert på vare (item)
    på tvers av status-kategorier (SUCCESS, REFUND, CANCELLED).
    """
    pivot = pd.pivot_table(
        df,
        values='price',
        index='item',
        columns='status',
        aggfunc='sum',
        fill_value=0.0
    )
    return pivot.round(2)


# ==============================================================================
# Task 4.4: Calculate refund ratio per item using Task 3 function
# ==============================================================================
def calculate_item_refund_ratios(df: pd.DataFrame) -> pd.DataFrame:
    """
    Beregner refund ratio per vare ved å telle antall salg (SUCCESS) og refusjoner (REFUND),
    og kaller compute_refund_ratio fra Oppgave 3.
    """
    # Teller antall transaksjoner per status for hver vare
    status_counts = df.groupby(['item', 'status']).size().unstack(fill_value=0)
    
    # Henter ut antall salg og refusjoner
    sales_count = status_counts['SUCCESS'].to_numpy() if 'SUCCESS' in status_counts.columns else np.zeros(len(status_counts))
    refund_count = status_counts['REFUND'].to_numpy() if 'REFUND' in status_counts.columns else np.zeros(len(status_counts))
    
    # Kaller funksjonen fra Oppgave 3.2
    ratios = compute_refund_ratio(sales_count, refund_count)
    
    result_df = pd.DataFrame({
        'item': status_counts.index,
        'sales_count': sales_count,
        'refund_count': refund_count,
        'refund_ratio': ratios
    })
    
    return result_df



In [8]:
# Task 05: Full Pipeline Automation med store-kolonne

def run_full_workflow(data_folder: str | Path, output_folder: str | Path = "output") -> dict[str, Path]:
    """
    Kjører hele datapipelinen automatisk og inkluderer butikkslokasjon (store).
    """
    data_path = Path(data_folder)
    out_path = Path(output_folder)
    out_path.mkdir(parents=True, exist_ok=True)

    # 1. Les metadata (.json) for å identifisere aktive kasser
    json_files = list(data_path.rglob("*.json"))
    active_registers = set()
    
    for json_file in json_files:
        try:
            content_str, _ = log_file_reader(json_file)
            json_data = json.loads(content_str)
            active_registers.update(metadata_recursive(json_data))
        except Exception as e:
            print(f"Advarsel: Kunne ikke lese JSON-fil {json_file}: {e}")

    # 2. Søk opp og les alle .log-filer, og legg til butikknavn (store)
    log_file_paths = directory_search_log_files(data_path)
    all_records = []

    for log_path in log_file_paths:
        file_text, _ = log_file_reader(log_path)
        records = parse_log_lines(file_text)
        
        # Henter butikknavnet fra mappen to nivåer opp (f.eks. library_cafe)
        store_name = log_path.parent.parent.name
        for r in records:
            r["store"] = store_name

        all_records.extend(records)

    if not all_records:
        raise FileNotFoundError(f"Ingen loggfiler ble funnet i {data_path.resolve()}.")

    # Filtrer kun transaksjoner fra aktive kasser dersom metadata finnes
    if active_registers:
        all_records = [r for r in all_records if r["register_id"] in active_registers]

    # 3. Bygg DataFrame og utfør datavask / korrigeringer
    df = create_transaction_dataframe(all_records)
    
    # Organiser kolonnene slik at 'store' står ved siden av 'register_id'
    cols = ['store', 'register_id', 'item', 'price', 'currency', 'status']
    df = df[[c for c in cols if c in df.columns]]

    # 3a. Vask 999.00 hardware-feil i priskolonnen med medianen
    df['price'] = clean_price_outliers(df['price'].to_numpy())
    
    # 3b. Valutakorrigering for NOK (pris * 10.0)
    df.loc[df['currency'] == 'NOK', 'price'] = df.loc[df['currency'] == 'NOK', 'price'] * 10.0

    # 3c. Avrund til 2 desimaler
    df['price'] = df['price'].round(2)

    # 4. Generer og eksporter de 4 rapportene
    cleaned_csv_path = out_path / "cleaned_transactions.csv"
    df.to_csv(cleaned_csv_path)

    hourly_df = resample_hourly_revenue(df)
    hourly_json_path = out_path / "hourly_revenue.json"
    hourly_df.to_json(hourly_json_path, orient="records", date_format="iso", indent=2)

    pivot_df = create_status_pivot_table(df)
    pivot_csv_path = out_path / "status_pivot.csv"
    pivot_df.to_csv(pivot_csv_path)

    refund_df = calculate_item_refund_ratios(df)
    refund_csv_path = out_path / "item_refund_ratios.csv"
    refund_df.to_csv(refund_csv_path, index=False)

    print("=== Pipeline fullført med suksess ===")
    print(f"Behandlet transaksjoner : {len(df)}")
    print(f"Filer lagret i mappen   : {out_path.resolve()}")

    return {
        "cleaned_transactions": cleaned_csv_path,
        "hourly_revenue": hourly_json_path,
        "status_pivot": pivot_csv_path,
        "refund_ratios": refund_csv_path
    }


# Kjøring av Oppgave 5
run_full_workflow(data_folder=Path.cwd(), output_folder="./output")

=== Pipeline fullført med suksess ===
Behandlet transaksjoner : 720
Filer lagret i mappen   : C:\Users\Bruker\Documents\INF201\Koding INF201\INF201_group_project\output


C:\Users\Bruker\AppData\Local\Temp\ipykernel_17568\1452599159.py:22: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.set_index('timestamp', inplace=True)
C:\Users\Bruker\AppData\Local\Temp\ipykernel_17568\1452599159.py:25: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.sort_index(inplace=True)


{'cleaned_transactions': WindowsPath('output/cleaned_transactions.csv'),
 'hourly_revenue': WindowsPath('output/hourly_revenue.json'),
 'status_pivot': WindowsPath('output/status_pivot.csv'),
 'refund_ratios': WindowsPath('output/item_refund_ratios.csv')}

## Discussion

The pipeline successfully processed a total of 720 valid transaction records across the active registers. All raw transactions were cleaned (removing extreme price glitches like 999.00), normalized for currency scaling (adjusting NOK values by a factor of 10.0), and aggregated into the required output reports. We can see that the lattes have the highest amount of refunds and croissants have the lowest! In status_pivot the sandwich gave the most success, but since it also had a high number of refunds it is high potential for improvement here.

## AI summary

Oppsummering av arbeid og feilsøking (Deliverable 2 – INF201)

1. **Implementering av arbeidsflyt (Task 5)**  
   Kombinert datainnlesting og analyse: Fullført koden for `run_full_workflow` som kobler sammen hele pipelinen: Rekursivt søk etter loggfiler med `Path` og uthenting av aktive kasse-ID-er fra `store_config.json`. Parsing av logglinjer til strukturert data og bygging av Pandas DataFrame. Fjerning av ekstremverdier/feilmeldinger (999.00) ved hjelp av NumPy. Valutakorrigering (multiplisering med 10.0 for transaksjoner i NOK). Generering og lagring av alle påkrevde rapportfiler (`cleaned_transactions.csv`, `hourly_revenue.json`, `status_pivot.csv`, og `item_refund_ratios.csv`).

2. **Versjonskontroll og Git-opprydding**  
   Mappestruktur og stibaner: Løst et større problem med lange, uhensiktsmessige Windows-mappestier (`Sales/Users/tutor5923/Documents/...`). Mappen `d2_data` ble flyttet helt opp til prosjektroten for å sikre ren og plattformuavhengig referering i koden. Opprydding i unødvendige filer og fjerning av overflødige testfiler.